# Sales Forecasting — Linear Regression (Project 2, Phase 1–3)
**Topic:** Linear Regression
**Dataset:** Kaggle — Sales Forecasting (rohitsahoo/sales-forecasting) + synthetic `Quantity`, `Discount`, `Profit`

**Important note on the data:** your real `train.csv` (9,800 rows) has 18 columns — it does **not** include `Quantity`, `Discount`, or `Profit` (that's a simplified export of the Superstore dataset). This notebook **generates them synthetically** using realistic Superstore-style rules, clearly marked wherever they're created. Every other column (`Sales`, `Category`, `Region`, etc.) is your real data — nothing else is fabricated.

**This version uses `Quantity`, `Discount`, and `Profit` directly as model features (feature extraction), as requested.** One honest flag up front: in the synthetic generation below, `Profit` is mathematically built *from* `Sales` (`Profit = Sales × margin_formula`). That means using `Profit` to predict `Sales` is technically **data leakage** — the model is partly shown a transformed version of the answer. We include it anyway per your request, but call this out clearly wherever it matters (feature list, VIF, and coefficient interpretation) so you understand — and can explain — exactly why the R² will look unusually high once `Profit` is added.

Every task below has a **What** (what the code does) and **Why** (why this step matters) explanation, followed by the code.


## Setup — Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, root_mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings 
warnings.filterwarnings("ignore")

sns.set(style="whitegrid")
%matplotlib inline


---
# PHASE 1: Data Collection

### Task 1 — Download the Sales Dataset
**What:** Download `train.csv` from https://www.kaggle.com/datasets/rohitsahoo/sales-forecasting and place it next to this notebook.
**Why:** This is the raw Superstore sales data every later task builds on.


In [ ]:
df = pd.read_csv("train.csv")
print(df.shape)
df.head()


### Add synthetic `Quantity`, `Discount`, `Profit` columns
**What:** Generate three new columns using rules typical of the real Superstore dataset:
- `Quantity`: random order quantity (1–9 units), independent of Sales.
- `Discount`: a category-weighted random discount level (Furniture/Office Supplies get discounted more often than Technology, matching real Superstore patterns).
- `Profit`: computed **from** `Sales` and `Discount` using a margin formula (`Profit = Sales × (base_margin − Discount × penalty) + noise`) — this mirrors how Profit is actually derived in the real Superstore dataset (heavier discounts erode margin, sometimes into a loss).

**Why:** You asked for a project that includes these three features. Since they're not in your actual file, we simulate them with realistic, non-arbitrary rules (seeded for reproducibility) rather than pure random noise, so the notebook still demonstrates believable relationships. We flag this clearly rather than presenting it as real data — that would misrepresent your dataset.


In [ ]:
rng = np.random.default_rng(42)

# Quantity: independent random order size
df['Quantity'] = rng.integers(1, 10, size=len(df))

# Discount: category-weighted realistic discount tiers
discount_tiers = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
category_discount_weights = {
    'Technology':       [0.35, 0.25, 0.15, 0.10, 0.05, 0.05, 0.03, 0.02],
    'Furniture':        [0.15, 0.15, 0.15, 0.15, 0.15, 0.10, 0.10, 0.05],
    'Office Supplies':  [0.20, 0.20, 0.15, 0.15, 0.10, 0.10, 0.05, 0.05],
}
df['Discount'] = df['Category'].apply(
    lambda cat: rng.choice(discount_tiers, p=category_discount_weights[cat])
)

# Profit: derived from Sales and Discount (mirrors real Superstore logic)
base_margin = 0.30
noise = rng.normal(0, 5, size=len(df))
df['Profit'] = df['Sales'] * (base_margin - df['Discount'] * 0.8) + noise
df['Profit'] = df['Profit'].round(2)

df[['Sales', 'Quantity', 'Discount', 'Profit']].describe()


---
# PHASE 2: Data Preprocessing (Sales Dataset)

### Task 1 — Handle missing values
**What:** Check for nulls/duplicates and fill or drop them. `Postal Code` is the only real column with missing values.
**Why:** Models can't train on NaNs, and filling with the mode is safe here since it's missing for only a small number of rows.


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

df['Postal Code'] = df['Postal Code'].fillna(df['Postal Code'].mode()[0])

df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')
df['Ship Date']  = pd.to_datetime(df['Ship Date'], dayfirst=True, errors='coerce')
print("\nMissing values after cleaning:", df.isnull().sum().sum())


### Task 6 — Split Sales Dataset into train (70%) and test (30%)
**What:** Split the raw rows into train/test **before** building engineered features.
**Why:** One of our features is a *target encoding* of `Product ID` (average sales level per product), which must be learned only from training rows to avoid leaking test information into the model — splitting first keeps evaluation honest.


In [ ]:
train_raw, test_raw = train_test_split(df, test_size=0.30, random_state=42)
print("Train rows:", train_raw.shape[0])
print("Test rows :", test_raw.shape[0])


### Feature engineering
**What:** Build the predictors for the model:
- One-hot encode `Category`, `Sub-Category`, `Region`, `Segment`, `Ship Mode`.
- Include `Quantity` and `Discount` directly (numeric, legitimate predictors of Sales — more units or a bigger discount genuinely changes the transaction value).
- Target-encode `Product ID` (average log-Sales per product, learned on train only) — a strong proxy for unit price.
- Log-transform `Sales` (`Sales_log = log1p(Sales)`) to fix right-skew.

**`Profit` is included as a predictor here, per your request.** Keep the leakage caveat above in mind: because `Profit` is derived from `Sales` in our synthetic formula, it will act almost like a disguised copy of the target, and R² will jump accordingly. In a real dataset where Profit is independently measured (not formula-derived from Sales), including it would be far more defensible.


In [ ]:
def build_features(frame, product_enc_map, global_mean):
    f = frame.copy()
    f['Sales_log'] = np.log1p(f['Sales'])
    cat_cols = ['Category', 'Sub-Category', 'Region', 'Segment', 'Ship Mode']
    dummies = pd.get_dummies(f[cat_cols], drop_first=True)
    f = pd.concat([f[['Sales', 'Sales_log', 'Product ID', 'Quantity', 'Discount', 'Profit']], dummies], axis=1)
    f['Product_enc'] = f['Product ID'].map(product_enc_map).fillna(global_mean)
    f = f.drop(columns=['Product ID'])
    return f.dropna()

# Learn Product ID -> average log(Sales) mapping from TRAIN ONLY
train_raw = train_raw.copy()
train_raw['Sales_log_tmp'] = np.log1p(train_raw['Sales'])
product_enc_map = train_raw.groupby('Product ID')['Sales_log_tmp'].mean()
global_mean = train_raw['Sales_log_tmp'].mean()

model_train = build_features(train_raw, product_enc_map, global_mean)
model_test  = build_features(test_raw, product_enc_map, global_mean)

# Align columns (test may be missing a rare dummy category)
model_train, model_test = model_train.align(model_test, join='left', axis=1, fill_value=0)

print(model_train.shape, model_test.shape)
model_train.head()


### Task 3 — Check for multicollinearity using VIF
**What:** Compute VIF for every candidate predictor, including `Quantity`, `Discount`, and `Profit`, using the training set.
**Why:** High VIF means a feature is redundant with the others. Watch `Profit` and `Discount` here especially — since `Profit` was built from `Sales` and `Discount`, we'd expect some correlation to show up.


In [ ]:
X_vif = model_train.drop(columns=['Sales', 'Sales_log']).astype(float)
X_vif = X_vif.assign(const=1)

vif_data = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data[vif_data['feature'] != 'const'].sort_values('VIF', ascending=False)
print(vif_data)


### Task 5 — Remove highly correlated features (VIF > 10)
**What:** Drop any feature whose VIF exceeds 10 from both train and test.
**Why:** VIF > 10 signals the feature is almost fully explainable by the other features already in the model.


In [ ]:
high_vif_features = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print("Dropping (VIF > 10):", high_vif_features)

model_train = model_train.drop(columns=high_vif_features)
model_test  = model_test.drop(columns=high_vif_features)

X_train = model_train.drop(columns=['Sales', 'Sales_log'])
y_train = model_train['Sales_log']
X_test  = model_test.drop(columns=['Sales', 'Sales_log'])
y_test  = model_test['Sales_log']

print(X_train.shape, X_test.shape)


---
# PHASE 3: Linear Regression (Sales Forecasting)

### Task 1 — Perform EDA: Plot sales vs each feature (scatter plots)
**What:** Scatter-plot the target against each numeric feature, including `Quantity`, `Discount`, and `Profit`.
**Why:** Confirms visually whether each predictor has a relationship with `Sales`. Watch the `Profit` panel especially — expect a strong, almost too-clean linear trend, since `Profit` was derived from `Sales` in the synthetic generation step.


In [ ]:
numeric_features = [c for c in X_train.columns if X_train[c].nunique() > 2]

fig, axes = plt.subplots(1, len(numeric_features), figsize=(5*len(numeric_features), 4))
if len(numeric_features) == 1:
    axes = [axes]
for ax, feat in zip(axes, numeric_features):
    ax.scatter(X_train[feat], y_train, alpha=0.3, s=10)
    ax.set_xlabel(feat)
    ax.set_ylabel("Sales (log)")
plt.tight_layout()
plt.show()


### Task 2 — Check linearity assumption using scatter plots
**What:** Interpret the scatter plots above.
**Why:** Linear regression assumes each predictor has a roughly straight-line relationship with the target. `Product_enc` should show a clear upward linear trend (it's already an average of the log-target, by construction). `Quantity` and `Discount` should show a milder but visible trend.


### Build the model once, reuse everywhere
**What:** Fit `LinearRegression` a single time; every diagnostic and evaluation step below reuses these predictions.
**Why:** Keeps every downstream check (Q-Q plot, residual plot, R²/RMSE, coefficients) consistent with the same trained model.


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_train = lr.predict(X_train)
y_pred_test  = lr.predict(X_test)

print("Model trained.")


### Task 3 — Check normality of residuals using Q-Q plot
**What:** Plot training residual quantiles against a theoretical normal distribution.
**Why:** Confidence intervals/p-values on coefficients are only valid if residuals are approximately normal.


In [ ]:
residuals = y_train - y_pred_train

stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()


### Task 4 — Check homoscedasticity using residual plot
**What:** Plot predicted values against residuals.
**Why:** Linear regression assumes constant error variance across the prediction range; a funnel shape would signal heteroscedasticity.


In [ ]:
plt.scatter(y_pred_train, residuals, alpha=0.3, s=10)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Sales (log)")
plt.ylabel("Residuals")
plt.title("Residual Plot (Homoscedasticity Check)")
plt.show()


### Task 5 — Build a simple Linear Regression model using `LinearRegression` from sklearn
**What:** (Already fit above.) This is the deliverable baseline model.
**Why:** Every other model (polynomial, Ridge, Lasso) is compared against this baseline.


### Task 6 — Evaluate model using R², RMSE
**What:** Compute R² and RMSE on the test set — on the log scale the model trained on, and converted back to $ for business interpretation.
**Why:** R² shows how much of Sales' variance is explained; RMSE gives a real, interpretable dollar error.


In [ ]:
r2 = r2_score(y_test, y_pred_test)
rmse = root_mean_squared_error(y_test, y_pred_test)

y_test_original = np.expm1(y_test)
y_pred_original = np.expm1(y_pred_test)
rmse_original = root_mean_squared_error(y_test_original, y_pred_original)

print(f"R²   (log scale)       : {r2:.4f}")
print(f"RMSE (log scale)       : {rmse:.4f}")
print(f"RMSE (original $ scale): {rmse_original:.2f}")


### Task 7 — Plot actual vs predicted sales
**What:** Scatter actual vs predicted Sales ($ scale) with a red diagonal reference line.
**Why:** Points hugging the diagonal indicate accurate predictions; systematic drift away from it reveals bias.


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test_original, y_pred_original, alpha=0.4, s=10)
plt.plot([y_test_original.min(), y_test_original.max()],
         [y_test_original.min(), y_test_original.max()], 'r--')
plt.xlabel("Actual Sales ($)")
plt.ylabel("Predicted Sales ($)")
plt.title("Actual vs Predicted Sales")
plt.show()


### Task 8 — Interpret coefficients (business meaning)
**What:** List each feature's coefficient and convert to an approximate % effect on Sales via `exp(coef) - 1` (valid since the target is log-transformed).
**Why:** Makes coefficients usable in a business report — e.g. "each extra unit of Quantity increases Sales by ~X%". Expect `Profit`'s coefficient to dominate the table — that's the leakage effect flagged earlier, not a genuine business insight, so don't present it as one without that caveat.


In [ ]:
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr.coef_
})
coef_df['Approx_%_effect_on_Sales'] = (np.exp(coef_df['Coefficient']) - 1) * 100
coef_df = coef_df.sort_values('Coefficient', ascending=False)
print(coef_df)


### Task 9 — Add interaction terms and polynomial features (`PolynomialFeatures`)
**What:** Expand the feature set to degree-2 polynomial/interaction terms.
**Why:** Tests whether combinations (e.g. Quantity × Discount) or curved relationships explain Sales better than a purely linear combination.


In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print("Original features:", X_train.shape[1])
print("Polynomial features (deg=2):", X_train_poly.shape[1])


### Task 10 — Build Polynomial Regression model (degree=2, 3)
**What:** Fit `LinearRegression` on the degree-2 and degree-3 polynomial feature sets.
**Why:** Degree 3 checks whether more complex interactions help further, or whether the model starts overfitting.


In [ ]:
results_poly = {}

for degree in [2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_p = poly.fit_transform(X_train)
    X_test_p  = poly.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_p, y_train)
    pred = model.predict(X_test_p)

    results_poly[degree] = {
        'R2': r2_score(y_test, pred),
        'RMSE': root_mean_squared_error(y_test, pred)
    }

results_poly


### Task 11 — Compare performance of polynomial regression vs linear
**What:** Tabulate R²/RMSE for linear vs degree-2 vs degree-3.
**Why:** Lets us decide, with numbers, whether the extra complexity of polynomial terms is worth it.


In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear', 'Poly (deg 2)', 'Poly (deg 3)'],
    'R2':   [r2, results_poly[2]['R2'], results_poly[3]['R2']],
    'RMSE': [rmse, results_poly[2]['RMSE'], results_poly[3]['RMSE']]
})
print(comparison)


### Task 12 — Use Ridge regression and tune alpha parameter
**What:** Fit Ridge across a range of `alpha` values, pick the best by test R².
**Why:** With many one-hot dummy features, Ridge's L2 penalty shrinks unstable/noisy coefficients and typically generalizes better.


In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]
ridge_scores = {}

for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train, y_train)
    pred = ridge.predict(X_test)
    ridge_scores[a] = r2_score(y_test, pred)

best_alpha_ridge = max(ridge_scores, key=ridge_scores.get)
print("Ridge R² by alpha:", ridge_scores)
print("Best alpha (Ridge):", best_alpha_ridge)


### Task 13 — Use Lasso regression and tune alpha parameter
**What:** Same sweep as Ridge, but with Lasso's L1 penalty.
**Why:** Lasso can shrink weak feature coefficients to exactly zero — a built-in feature-selection check.


In [ ]:
lasso_scores = {}

for a in alphas:
    lasso = Lasso(alpha=a, max_iter=10000)
    lasso.fit(X_train, y_train)
    pred = lasso.predict(X_test)
    lasso_scores[a] = r2_score(y_test, pred)

best_alpha_lasso = max(lasso_scores, key=lasso_scores.get)
print("Lasso R² by alpha:", lasso_scores)
print("Best alpha (Lasso):", best_alpha_lasso)


### Task 14 — Compare Linear, Ridge, and Lasso regression results
**What:** Side-by-side R²/RMSE for all three models at their best settings.
**Why:** Final head-to-head to pick the model that generalizes best, not just the one that fits training data best.


In [ ]:
final_ridge = Ridge(alpha=best_alpha_ridge).fit(X_train, y_train)
final_lasso = Lasso(alpha=best_alpha_lasso, max_iter=10000).fit(X_train, y_train)

comparison_final = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge', 'Lasso'],
    'R2': [
        r2_score(y_test, lr.predict(X_test)),
        r2_score(y_test, final_ridge.predict(X_test)),
        r2_score(y_test, final_lasso.predict(X_test)),
    ],
    'RMSE': [
        root_mean_squared_error(y_test, lr.predict(X_test)),
        root_mean_squared_error(y_test, final_ridge.predict(X_test)),
        root_mean_squared_error(y_test, final_lasso.predict(X_test)),
    ]
})
print(comparison_final)


### Task 15 — Perform cross-validation (5-fold) for all models
**What:** Run 5-fold CV for Linear, Ridge, Lasso on the combined train+test feature set, reporting mean and std of R².
**Why:** A single split can be lucky/unlucky; 5-fold CV gives a more robust performance estimate.


In [ ]:
X_all = pd.concat([X_train, X_test], axis=0)
y_all = pd.concat([y_train, y_test], axis=0)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=best_alpha_ridge),
    'Lasso': Lasso(alpha=best_alpha_lasso, max_iter=10000)
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_all, y_all, cv=kf, scoring='r2')
    cv_results[name] = (scores.mean(), scores.std())

for name, (mean_r2, std_r2) in cv_results.items():
    print(f"{name}: mean R² = {mean_r2:.4f} (+/- {std_r2:.4f})")


---
## Summary

- **Data transparency:** `Quantity`, `Discount`, `Profit` were not in your real dataset and were generated synthetically with realistic Superstore-style rules — clearly flagged, not passed off as real data.
- **What predicts Sales:** product category/sub-category, region, segment, ship mode, `Quantity`, `Discount`, `Profit`, and a train-only target encoding of `Product ID` (proxy for unit price).
- **Leakage caveat (important for your report):** `Profit` was included as a predictor per your request, but it was mathematically built from `Sales` during data generation. This means R² will look unusually high mainly *because of* `Profit`, not because the model has learned a genuinely new business relationship. If your grader asks about this, the honest answer is: "Profit was included because it was requested, but it's a form of data leakage since Profit is derived from Sales — in a real dataset with independently-recorded Profit, this would be a legitimate feature."
- Diagnostics (Q-Q plot, residual plot) check whether linear regression's assumptions hold.
- Polynomial, Ridge, and Lasso variants were compared against the linear baseline and validated with 5-fold cross-validation.

**Note:** Since `Quantity`/`Discount`/`Profit` are synthetic, don't present the exact R²/coefficient numbers as findings about your real business — present the *workflow* as the deliverable, and be upfront that these three columns were simulated to fulfill the assignment requirement.
